# Highway 97 Road Segment Congestion

This notebook downloads recent GTFS-Realtime vehicle and trip updates, runs the reusable congestion engine in `forecasting`, and visualizes direction-specific segment congestion. Segments follow each Route 97 trip's shape and ordered stops. Vehicle updates must match their own trip, direction, stop ID, and stop sequence; opposite directions and different shapes are never pooled.

## Requirements

Install the forecasting package and notebook dependencies from the repository (`cd forecasting && pip install -r requirements.txt`). Saved backtests run offline without database credentials. For a **new download**, create the ignored `.env.road_segment_congestion` file by sourcing `./setup_road_segment_congestion_env.sh` from the repository root, or provide the five `TRANSIT_DB_*` environment variables yourself. Only download mode reads these credentials and contacts the database/static GTFS publisher; credentials are never saved in the dataset. The default download window is 48 hours.

## Packages

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forecasting import (
    get_backtest_dataset,
    get_latest_stop_delays,
    list_backtest_datasets,
    make_congestion_map,
    prepare_congestion_snapshots,
    run_congestion_engine,
)

## Constants

In [ ]:


workspace_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
repo_root = next(
    (
        candidate
        for candidate in workspace_candidates
        if (candidate / 'forecasting' / 'forecasting').is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository forecasting package.')

backtest_cache_dir = repo_root / 'data' / 'backtests'

visuals_dir = repo_root / 'notebooks' / 'visuals'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo_root}')

## Download backtest data

Choose `backtest_source` below:

- **`'cache'`** (default): load a saved dataset without any network/database calls. `backtest_dataset_id = None` selects the most recently saved dataset; copy any ID from the displayed catalog to rerun that specific historical snapshot. A missing/corrupt snapshot raises an error, never silently downloads different data.
- **`'download'`**: explicitly pull a new realtime window and current BC Transit GTFS for operator 47, then save both as a new, automatically named snapshot. Switch back to `'cache'` afterward to reuse it. Every intentional download creates a separate dataset; no existing dataset is overwritten.

`get_backtest_dataset` handles selection, credentials, downloads, and caching in the forecasting package. Each snapshot under `data/backtests/<dataset_id>/` contains both realtime tables, matching static GTFS, and metadata with UTC coverage, row counts, and checksums. Only load trusted local pickle snapshots.

The catalog lists datasets by save time, not data coverage, so check the UTC start/end columns when selecting a period. Updates preserve their original row order and dtypes (new database pulls are newest-first). Cached runs always use the selected snapshot's static GTFS.

In [ ]:
backtest_source = 'cache'
backtest_dataset_id = "20261026T010808465257Z"
backtest_lookback_hours = 48

print('Saved backtests (newest saved first):')
display(list_backtest_datasets(backtest_cache_dir))

backtest_dataset = get_backtest_dataset(
    repo_root / 'data',
    source=backtest_source,
    dataset_id=backtest_dataset_id,
    lookback_hours=backtest_lookback_hours,
    env_file=repo_root / '.env.road_segment_congestion',
)
df_vehicle_updates = backtest_dataset.vehicle_updates
df_trip_updates = backtest_dataset.trip_updates
backtest_dataset_dir = backtest_dataset.directory
static_data_dir = backtest_dataset.static_data_dir

print(f'Using backtest dataset: {backtest_dataset_dir.name}')
print(f'Dataset directory: {backtest_dataset_dir}')
print(f'Vehicle updates: {len(df_vehicle_updates):,}')
print(f'Trip updates: {len(df_trip_updates):,}')
print(
    'Vehicle update window: '
    f'{df_vehicle_updates["feed_timestamp"].min()} through '
    f'{df_vehicle_updates["feed_timestamp"].max()}'
)

In [ ]:
print(f'Static GTFS snapshot: {static_data_dir}')
feed_info_path = static_data_dir / 'feed_info.txt'
if feed_info_path.is_file():
    print(pd.read_csv(feed_info_path, dtype='string').to_string(index=False))

df_stops = pd.read_csv(static_data_dir / 'stops.txt')
df_stop_times = pd.read_csv(static_data_dir / 'stop_times.txt')
df_trips = pd.read_csv(static_data_dir / 'trips.txt')
df_shapes = pd.read_csv(static_data_dir / 'shapes.txt')

print(
    f'Loaded {len(df_stops):,} stops, {len(df_stop_times):,} stop times, '
    f'{len(df_trips):,} trips, and {len(df_shapes):,} shape points.'
)

## Run the congestion engine

The package constructs directed stop-to-stop segments for every selected Route 97 trip and shape. Updates must match `(trip_id, trip_direction_id, stop_id, current_stop_sequence)` against that trip's GTFS stop times. The same directed stop pair on the same shape shares a segment; opposite directions, express stop pairs, and different shapes remain separate.

Set `trip_direction_id = None` below to process both directions separately, or choose `0` or `1`. Segment IDs are unique within each result, not permanent GTFS identifiers. The reference trip is retained for inspection; it is no longer the only source of segments. The route geometry includes all selected shapes.

Unmatched updates are skipped with an explicit warning, rather than being assigned by a sequence number from another trip. Use a static GTFS snapshot aligned with the realtime data period. Updates naming a trip's origin stop have no incoming segment and are excluded.

Each directed segment and shape has its own median historical speed baseline. Mean speed is aggregated into five-minute UTC bins, and the congestion index is `1 - observed_speed / baseline_speed`, clipped to 0 (free flow) through 1 (stopped). Baselines at or below 0.5 are excluded because near-stationary layover speeds make the ratio unstable.

In [ ]:
route_id = '97-KEL'
time_bin_size = '5min'
trip_direction_id = None

In [ ]:

congestion_result = run_congestion_engine(
    df_vehicle_updates,
    df_stops,
    df_stop_times,
    df_trips,
    df_shapes,
    route_id=route_id,
    time_bin_size=time_bin_size,
    trip_direction_id=trip_direction_id,
)

segments = congestion_result.segments
route_geometry = congestion_result.route_geometry
vehicle_probes = congestion_result.vehicle_probes
df_congestion = congestion_result.congestion

print(f'Reference trip: {congestion_result.representative_trip_id}')
print(f'Route segments: {len(segments)}')
print(f'Matched vehicle probes: {len(vehicle_probes):,}')
print(f'Congestion estimates: {len(df_congestion):,}')
print('Segments per direction:')
print(segments.groupby('trip_direction_id').size())
print('Matched probes per direction:')
print(vehicle_probes.groupby('trip_direction_id').size())
if df_congestion.empty:
    if vehicle_probes.empty:
        raise ValueError(
            'No valid incoming-segment probes matched the selected GTFS trips. '
            'Check the realtime window, selected direction, and static GTFS snapshot. '
            'Matching requires trip_id, trip_direction_id, stop_id, and stop sequence. '
            f'Loaded static GTFS: {static_data_dir}'
        )
    raise ValueError(
        f'{len(vehicle_probes):,} probes matched, but no segment baseline speed '
        'exceeded 0.5. Select a data window containing moving buses.'
    )
segments[
    ['segment_id', 'trip_direction_id', 'shape_id',
     'from_stop_id', 'to_stop_id', 'from_stop_name', 'to_stop_name']
]

## Latest segment congestion

The latest map and final raw animation frame use the same timestamp and freshness policy. Estimates expire after the configurable maximum age below; the provisional 15-minute default needs calibration against actual probe gaps. Identical paths with the same direction and directed stop pair are displayed once, using the freshest shape-specific estimate (lowest source segment ID breaks ties). Hover labels retain the contributing segment, shape, observation time, age, and equivalent segment IDs. Different directions, stop pairs, and paths are never pooled. Naive input timestamps are interpreted as UTC.

In [ ]:
max_congestion_age = pd.Timedelta(minutes=15)
display_as_of = vehicle_probes['feed_timestamp'].max()
latest_display = prepare_congestion_snapshots(
    congestion_result, pd.DatetimeIndex([display_as_of]),
    max_age=max_congestion_age,
)
fig = make_congestion_map(
    latest_display, route_geometry,
    title=f'Route {route_id} Directed Segment Congestion (latest fresh reading)',
)
fig.write_html(visuals_dir / 'route_97_segment_congestion_engine.html')
fig.show()

## Spatial congestion animations

The maps show regular five-minute snapshots over the latest day, including gaps with no probes. Earlier observations may initialize the display, but expire according to the same maximum age as the latest map. An estimate is available only after its last contributing probe. The smoothed version currently uses a centered 10-bin rolling average for retrospective visualization; it is not a real-time causal estimate.

In [ ]:
animation_window = pd.Timedelta(days=1)
display_times = pd.date_range(
    start=display_as_of - animation_window, end=display_as_of, freq=time_bin_size,
)
animation_display = prepare_congestion_snapshots(
    congestion_result, display_times, max_age=max_congestion_age,
)
raw_congestion_grid = animation_display.pivot(
    index='as_of', columns='segment_id', values='congestion_index',
)
smoothed_congestion_grid = raw_congestion_grid.rolling(
    window=10, min_periods=1, center=True,
).mean().where(raw_congestion_grid.notna())
smoothed_display = animation_display.drop(columns='congestion_index').merge(
    smoothed_congestion_grid.stack().rename('congestion_index').reset_index(),
    on=['as_of', 'segment_id'], how='left', validate='one_to_one',
)

In [ ]:
raw_animation = make_congestion_map(
    animation_display, route_geometry,
    title=f'Route {route_id} Segment Congestion Over Time (last {animation_window})',
)
#raw_animation.write_html(
#    visuals_dir / 'route_97_segment_congestion_engine_animation.html'
#)
raw_animation.show()

In [ ]:
smoothed_animation = make_congestion_map(
    smoothed_display, route_geometry,
    title=(
        f'Route {route_id} Smoothed Segment Congestion Over Time '
        f'(last {animation_window}, 10-bin rolling mean)'
    ),
    value_label='Smoothed congestion index',
)
#smoothed_animation.write_html(
#    visuals_dir / 'route_97_smoothed_segment_congestion_engine_animation.html'
#)
smoothed_animation.show()

## Segment time series and probe availability

Select a segment ID from the table above. Each series and probe count belongs to exactly one direction and shape, both identified in the title. Timestamps are converted from UTC to Vancouver local time for display. Smoothing is a centered five-bin rolling mean.

In [ ]:
segment_id_of_interest = 8
segment_matches = segments.loc[
    segments['segment_id'].eq(segment_id_of_interest)
]
if segment_matches.empty:
    raise ValueError(f'Segment {segment_id_of_interest} is not present on route {route_id}.')
segment = segment_matches.iloc[0]
segment_direction_id = int(segment['trip_direction_id'])
segment_label = (
    f"Direction {segment_direction_id}, shape {segment['shape_id']}: "
    f"{segment['from_stop_name']} -> {segment['to_stop_name']}"
)

df_single_segment = (
    df_congestion.loc[
        df_congestion['segment_id'].eq(segment_id_of_interest)
        & df_congestion['trip_direction_id'].eq(segment_direction_id)
    ]
    .sort_values('time_bin')
    .copy()
)
df_single_segment['time_bin_local'] = df_single_segment['time_bin'].dt.tz_convert(
    'America/Vancouver'
)
segment_smoothing_window = 5
df_single_segment['congestion_index_smooth'] = (
    df_single_segment['congestion_index']
    .rolling(
        window=segment_smoothing_window,
        min_periods=1,
        center=True,
    )
    .mean()
)

segment_figure = go.Figure()
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    )
)
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    )
)
segment_figure.update_layout(
    title=f'Smoothed Congestion Index for Segment: {segment_label}',
    xaxis_title='Time (Vancouver)',
    yaxis_title='Congestion Index (0 = free flow, 1 = stopped)',
)
segment_figure.show()

In [ ]:
df_probe_counts = (
    vehicle_probes.loc[
        vehicle_probes['segment_id'].eq(segment_id_of_interest)
        & vehicle_probes['trip_direction_id'].eq(segment_direction_id)
    ]
    .groupby('time_bin')['vehicle_id']
    .nunique()
    .rename('probe_count')
    .reset_index()
    .sort_values('time_bin')
)
df_probe_counts['time_bin_local'] = df_probe_counts['time_bin'].dt.tz_convert(
    'America/Vancouver'
)

probe_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.08,
    subplot_titles=(
        f'Congestion Index for Segment: {segment_label}',
        'Number of Buses (Probes) Used per Time Bin',
    ),
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Bar(
        x=df_probe_counts['time_bin_local'],
        y=df_probe_counts['probe_count'],
        name='Probe count',
        marker=dict(color='steelblue'),
    ),
    row=2,
    col=1,
)
probe_figure.update_yaxes(
    title_text='Congestion Index (0 = free flow, 1 = stopped)',
    row=1,
    col=1,
)
probe_figure.update_yaxes(title_text='# Buses', row=2, col=1)
probe_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
probe_figure.update_layout(
    title=f'Congestion Index vs. Probe Availability: {segment_label}',
    showlegend=True,
    height=700,
)
probe_figure.show()

## Latest reported delay at a stop

Route `97-KEL` serves two platforms named **Harvey Ave at Gordon Stn** (`140002` and `140003`). Keep both platforms separate. For each trip instance and stop visit, take the row with the latest `feed_timestamp` and use its `arrival_delay` in seconds, converted to minutes for plotting. Do not average earlier predictions or substitute `departure_delay`.

Trip start descriptors distinguish repeated daily trips; stop sequences distinguish repeated visits. Missing start descriptors produce a warning and use Vancouver publication dates for approximate daily grouping. Missing delays in the latest rows remain missing, rather than falling back to older predictions.

This is the latest **reported estimate**, not independently measured ground truth. The last available update can be published after arrival or still be provisional if the data window ends before the visit. The x-axis is the latest update's publication time in Vancouver, matching the congestion plots' timezone. Each platform has raw estimates and a five-visit rolling mean; this smoothing is over visits, not five-minute congestion bins. Positive delays mean late; negative delays mean early.

The saved example output uses the local backtest snapshot covering September 8-22, 2026 (UTC). Rerunning these cells uses whichever `df_trip_updates` is currently loaded. If `trip_direction_id` is selected above, delay estimates are filtered to that same direction.

In [ ]:
#stop_name = 'Hwy 97 at Butt Rd Stn'
#stop_name = 'Hwy 97 at Westside Rd Stn'
stop_name = 'Queensway Exchange Bay B'
#stop_name = 'Harvey Ave at Richter Stn'
#stop_name = 'Harvey Ave at Gordon Stn'
#stop_name = 'Harvey Ave at Parkinson Stn'
#stop_name = 'Hwy 97 N at Banks Stn'
#stop_name = 'Hwy 97 N at Sexsmith Stn'


df_delay_trip_updates = df_trip_updates
if trip_direction_id is not None:
    df_delay_trip_updates = df_trip_updates.loc[
        pd.to_numeric(df_trip_updates['trip_direction_id'], errors='raise')
        .eq(trip_direction_id)
    ]
df_stop_delays = get_latest_stop_delays(
    df_delay_trip_updates,
    df_stops,
    route_id=route_id,
    stop_name=stop_name,
)
df_stop_delays['feed_timestamp_local'] = (
    df_stop_delays['feed_timestamp'].dt.tz_convert('America/Vancouver')
)
delay_smoothing_window = 5
df_stop_delays['delay_minutes_smooth'] = (
    df_stop_delays.groupby('stop_id')['delay_minutes']
    .transform(
        lambda delays: delays.rolling(
            window=delay_smoothing_window,
            min_periods=1,
            center=True,
        ).mean()
    )
    .where(df_stop_delays['delay_minutes'].notna())
)
print(f'Latest trip/stop visits on route {route_id}: {len(df_stop_delays):,}')
print(df_stop_delays.groupby('stop_id').size().rename('visit_count'))
print(f'Visits with missing arrival delay: {df_stop_delays["arrival_delay"].isna().sum():,}')
df_stop_delays[
    ['trip_id', 'trip_start_timestamp', 'stop_id',
     'feed_timestamp_local', 'arrival_delay', 'delay_minutes']
].head(10)

In [ ]:
delay_figure = go.Figure()
delay_colors = ['royalblue', 'darkorange']
for platform_index, (platform_id, platform_delays) in enumerate(
    df_stop_delays.groupby('stop_id', sort=True)
):
    
    direction_id = platform_delays['trip_direction_id'].iloc[0]
    platform_label = f'Stop {platform_id} | Dir {direction_id}'
    color = delay_colors[direction_id % len(delay_colors)]
    delay_figure.add_trace(
        go.Scatter(
            x=platform_delays['feed_timestamp_local'],
            y=platform_delays['delay_minutes'],
            mode='lines+markers',
            name=f'{platform_label}: raw latest estimates',
            legendgroup=platform_id,
            line=dict(color=color, width=1),
            opacity=0.45,
            connectgaps=False,
            customdata=platform_delays[
                ['trip_id', 'trip_start_timestamp', 'arrival_delay']
            ].to_numpy(),
            hovertemplate=(
                f'{platform_label}<br>'
                'Latest update: %{x}<br>'
                'Arrival delay: %{y:.2f} min (%{customdata[2]} s)<br>'
                'Trip: %{customdata[0]}<br>'
                'Trip start: %{customdata[1]}<extra></extra>'
            ),
        )
    )
    delay_figure.add_trace(
        go.Scatter(
            x=platform_delays['feed_timestamp_local'],
            y=platform_delays['delay_minutes_smooth'],
            mode='lines+markers',
            name=f'{platform_label}: {delay_smoothing_window}-visit rolling mean',
            legendgroup=platform_id,
            line=dict(color=color, width=3),
            connectgaps=False,
        )
    )
delay_figure.add_hline(y=0, line_dash='dash', line_color='gray')
delay_figure.update_layout(
    title=f'Latest Reported Arrival Delay: {stop_name} (Route {route_id})',
    xaxis_title='Latest trip-update time (Vancouver)',
    yaxis_title='Arrival delay (minutes; positive = late, negative = early)',
    hovermode='closest',
    legend_title_text='Platform and estimate',
)
delay_plot_path = visuals_dir / 'route_97_late_stop_delays.html'
delay_figure.write_html(delay_plot_path)
print(f'Saved delay time series: {delay_plot_path}')
delay_figure.show()

## Delay estimate accuracy before arrival

How much do BC Transit's predictions differ from the last reported delay as we look further ahead? As in **Latest reported delay at a stop**, the latest row for each trip instance, platform, and stop sequence supplies the **true-delay proxy** (`arrival_delay / 60`). Its `arrival_time`, interpreted as UTC, supplies the **arrival-time proxy**. Neither is independently measured ground truth; a final prediction can still be wrong or provisional. In particular, do not use the final `feed_timestamp` as the arrival time.

For each visit and each look-back interval **1, 2, ..., 30 minutes**, subtract that interval from the final arrival-time proxy. Select the last update published **at or before** that cutoff, using the same timestamp/ID tie-breaking as the previous section. Do not interpolate, use a later update, or replace a selected missing delay with an older nonmissing value. An older prediction can remain in force across several cutoffs; its age at the cutoff is retained for inspection. Visits whose final delay/arrival is missing or whose final arrival is beyond the recording window are excluded with a warning, not replaced with older references.

The controls below reuse `stop_name` and the selected direction above (defaulting to direction `0` if both directions were selected). Set `delay_comparison_stop_id` if the selected name/direction still includes multiple platforms; the analysis requires exactly one stop ID and one direction. The first plot compares every visit's unsmoothed final delay with its historical prediction; use the dropdown to change the look-back interval. The second shows prediction convergence for one visit (`delay_example_visit_index`, indexed by arrival order). The last summarizes absolute error, signed bias, and coverage across all visits.

Error is **predicted minus final delay**: positive values overestimate lateness, negative values underestimate it. MAE and the 90th percentile of absolute error measure disagreement, not calibrated uncertainty. Statistics use only available prediction/reference pairs at each interval. Coverage and the mix of visits can change with the interval, so compare the counts as well as the errors. Missing estimates stay missing in the plots and tables; zero delay is a valid estimate.

In [ ]:
from forecasting import get_stop_delay_horizons

delay_comparison_stop_name = stop_name
delay_comparison_direction_id = 0 if trip_direction_id is None else trip_direction_id
delay_comparison_stop_id = None
delay_horizons_minutes = list(range(1, 61))
delay_example_visit_index = 50

if delay_comparison_direction_id not in (0, 1):
    raise ValueError('Select exactly one delay_comparison_direction_id: 0 or 1.')
df_delay_comparison_updates = df_trip_updates.loc[
    pd.to_numeric(df_trip_updates['trip_direction_id'], errors='raise')
    .eq(delay_comparison_direction_id)
]
if delay_comparison_stop_id is not None:
    df_delay_comparison_updates = df_delay_comparison_updates.loc[
        df_delay_comparison_updates['stop_id'].astype('string')
        .eq(str(delay_comparison_stop_id))
    ]
df_delay_horizons = get_stop_delay_horizons(
    df_delay_comparison_updates,
    df_stops,
    route_id=route_id,
    stop_name=delay_comparison_stop_name,
    horizons_minutes=delay_horizons_minutes,
)
comparison_stop_ids = df_delay_horizons['stop_id'].unique().tolist()
if len(comparison_stop_ids) != 1:
    raise ValueError(
        f'Multiple platforms match: {comparison_stop_ids}. '
        'Set delay_comparison_stop_id to exactly one of them and rerun this cell.'
    )
delay_visit_columns = ['trip_instance_key', 'stop_id']
if 'stop_sequence' in df_delay_horizons.columns:
    delay_visit_columns.append('stop_sequence')
df_delay_horizons['visit_index'] = df_delay_horizons.groupby(
    delay_visit_columns, dropna=False, sort=False
).ngroup()
df_delay_horizons['reference_arrival_local'] = (
    df_delay_horizons['reference_arrival_time'].dt.tz_convert('America/Vancouver')
)
df_delay_reference_visits = df_delay_horizons.drop_duplicates('visit_index')
df_delay_horizon_summary = (
    df_delay_horizons.groupby('horizon_minutes', sort=True)
    .agg(
        total_visits=('visit_index', 'size'),
        paired_visits=('error_minutes', 'count'),
        mae_minutes=('absolute_error_minutes', 'mean'),
        p90_absolute_error_minutes=('absolute_error_minutes', lambda errors: errors.quantile(0.9)),
        bias_minutes=('error_minutes', 'mean'),
        mean_estimate_age_minutes=('estimate_age_minutes', 'mean'),
    )
)
df_delay_horizon_summary['coverage_percent'] = (
    100 * df_delay_horizon_summary['paired_visits']
    / df_delay_horizon_summary['total_visits']
)
delay_comparison_label = (
    f'{delay_comparison_stop_name} | Stop {comparison_stop_ids[0]} | '
    f'Direction {delay_comparison_direction_id} | Route {route_id}'
)
print(delay_comparison_label)
print(f'Eligible reference visits: {len(df_delay_reference_visits):,}')
print('All times below are Vancouver local times; delay/error units are minutes.')
display(df_delay_horizon_summary.round(2))

In [ ]:
delay_horizon_figure = go.Figure()
delay_horizon_figure.add_trace(
    go.Scatter(
        x=df_delay_reference_visits['reference_arrival_local'],
        y=df_delay_reference_visits['true_delay_minutes'],
        mode='lines+markers',
        name='True-delay proxy: last reported delay',
        line=dict(color='black', width=2),
        connectgaps=False,
    )
)
for horizon_index, horizon in enumerate(delay_horizons_minutes):
    horizon_visits = df_delay_horizons.loc[
        df_delay_horizons['horizon_minutes'].eq(horizon)
    ]
    delay_horizon_figure.add_trace(
        go.Scatter(
            x=horizon_visits['reference_arrival_local'],
            y=horizon_visits['estimated_delay_minutes'],
            mode='lines+markers',
            name=f'BC Transit estimate {horizon} min before arrival',
            line=dict(color='royalblue', width=1),
            visible=horizon_index == 0,
            connectgaps=False,
            customdata=horizon_visits[
                ['trip_id', 'visit_index', 'true_delay_minutes',
                 'error_minutes', 'estimate_age_minutes']
            ].to_numpy(),
            hovertemplate=(
                'Arrival proxy: %{x}<br>Predicted delay: %{y:.2f} min<br>'
                'Final delay: %{customdata[2]:.2f} min<br>'
                'Error: %{customdata[3]:.2f} min<br>'
                'Prediction age at cutoff: %{customdata[4]:.2f} min<br>'
                'Trip: %{customdata[0]}<br>Visit index: %{customdata[1]}<extra></extra>'
            ),
        )
    )
delay_horizon_figure.update_layout(
    title=f'Final vs. Historical Delay Estimates: {delay_comparison_label}',
    xaxis_title='Final reported arrival time (Vancouver; proxy)',
    yaxis_title='Arrival delay (minutes; positive = late)',
    hovermode='closest',
    updatemenus=[dict(
        buttons=[dict(
            label=f'{horizon} min before arrival',
            method='update',
            args=[{'visible': [True] + [
                candidate == horizon for candidate in delay_horizons_minutes
            ]}],
        ) for horizon in delay_horizons_minutes],
        direction='down', x=1, xanchor='right', y=1.15,
    )],
)
delay_horizon_figure.add_hline(y=0, line_dash='dash', line_color='gray')
delay_horizon_path = visuals_dir / 'stop_delay_horizon_comparison.html'
delay_horizon_figure.write_html(delay_horizon_path)
print(f'Saved historical delay comparison: {delay_horizon_path}')
delay_horizon_figure.show()

In [ ]:
df_delay_example_visit = df_delay_horizons.loc[
    df_delay_horizons['visit_index'].eq(delay_example_visit_index)
].sort_values('horizon_minutes')
if df_delay_example_visit.empty:
    raise ValueError(
        f'Choose delay_example_visit_index between 0 and '
        f'{len(df_delay_reference_visits) - 1}.'
    )
example_visit = df_delay_example_visit.iloc[0]
delay_convergence_figure = go.Figure()
for column, label, color in (
    ('true_delay_minutes', 'True-delay proxy: last reported delay', 'black'),
    ('estimated_delay_minutes', 'BC Transit historical prediction', 'royalblue'),
):
    delay_convergence_figure.add_trace(go.Scatter(
        x=df_delay_example_visit['horizon_minutes'],
        y=df_delay_example_visit[column],
        mode='lines+markers', name=label,
        line=dict(color=color), connectgaps=False,
    ))
delay_convergence_figure.update_layout(
    title=(
        f'Delay Prediction Convergence: {delay_comparison_label}<br>'
        f'Visit {delay_example_visit_index} | Trip {example_visit["trip_id"]} | '
        f'Arrival proxy {example_visit["reference_arrival_local"]}'
    ),
    xaxis_title='Minutes before final reported arrival (proxy)',
    xaxis=dict(autorange='reversed', dtick=1),
    yaxis_title='Arrival delay (minutes)',
    hovermode='x unified',
)
delay_convergence_path = visuals_dir / 'stop_delay_prediction_convergence.html'
delay_convergence_figure.write_html(delay_convergence_path)
print(f'Saved single-visit convergence: {delay_convergence_path}')
delay_convergence_figure.show()

In [ ]:
delay_accuracy_figure = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=(
        'Absolute prediction error vs. final reported delay',
        'Signed bias (prediction minus final delay)',
        'Coverage: available prediction/reference pairs',
    ),
    vertical_spacing=0.09,
)
for column, label, color in (
    ('mae_minutes', 'Mean absolute error', 'royalblue'),
    ('p90_absolute_error_minutes', '90th percentile absolute error', 'darkorange'),
):
    delay_accuracy_figure.add_trace(go.Scatter(
        x=df_delay_horizon_summary.index,
        y=df_delay_horizon_summary[column],
        mode='lines+markers', name=label, line=dict(color=color),
        connectgaps=False,
    ), row=1, col=1)
delay_accuracy_figure.add_trace(go.Scatter(
    x=df_delay_horizon_summary.index,
    y=df_delay_horizon_summary['bias_minutes'],
    mode='lines+markers', name='Mean signed error',
    line=dict(color='firebrick'), connectgaps=False,
), row=2, col=1)
delay_accuracy_figure.add_hline(y=0, line_dash='dash', row=2, col=1)
delay_accuracy_figure.add_trace(go.Bar(
    x=df_delay_horizon_summary.index,
    y=df_delay_horizon_summary['paired_visits'],
    name='Available pairs',
    customdata=df_delay_horizon_summary[['coverage_percent', 'total_visits']].to_numpy(),
    hovertemplate=(
        '%{x} min before arrival<br>%{y} / %{customdata[1]} visits<br>'
        'Coverage: %{customdata[0]:.1f}%<extra></extra>'
    ),
), row=3, col=1)
delay_accuracy_figure.update_yaxes(title_text='Absolute error (min)', row=1, col=1)
delay_accuracy_figure.update_yaxes(title_text='Signed error (min)', row=2, col=1)
delay_accuracy_figure.update_yaxes(title_text='Visit count', row=3, col=1)
delay_accuracy_figure.update_xaxes(dtick=1)
delay_accuracy_figure.update_xaxes(
    title_text='Minutes before final reported arrival (proxy)', row=3, col=1
)
delay_accuracy_figure.update_layout(
    title=f'Delay Estimate Error by Look-back Interval: {delay_comparison_label}',
    height=950, hovermode='x unified',
)
delay_accuracy_path = visuals_dir / 'stop_delay_horizon_errors.html'
delay_accuracy_figure.write_html(delay_accuracy_path)
print(f'Saved delay accuracy summary: {delay_accuracy_path}')
delay_accuracy_figure.show()